In [1]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

# Load the images
image1 = cv2.imread('foto1A.jpg')
image2 = cv2.imread('foto1B.jpg')
Images = []

ImageNames = os.listdir("stitching")
ImageNames_Split = [[(os.path.splitext(os.path.basename(ImageName))[0]), ImageName] for ImageName in ImageNames]
ImageNames_Split = sorted(ImageNames_Split, key=lambda x:x[0])
ImageNames_Sorted = [ImageNames_Split[i][1] for i in range(len(ImageNames_Split))]
for i in range(len(ImageNames_Sorted)):                     # Getting all image's name present inside the folder.
    ImageName = ImageNames_Sorted[i]
    InputImage = cv2.imread("stitching" + "/" + ImageName)  # Reading images one by one.
    Images.append(InputImage) 
print((Images))
    
    
def warp_images(img1, img2, H):
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    corners1 = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)
    corners2 = np.float32([[0, 0], [0, h2], [w2, h2], [w2, 0]]).reshape(-1, 1, 2)
    warped_corners2 = cv2.perspectiveTransform(corners2, H)

    corners = np.concatenate((corners1, warped_corners2), axis=0)
    [xmin, ymin] = np.int32(corners.min(axis=0).ravel() - 0.5)
    [xmax, ymax] = np.int32(corners.max(axis=0).ravel() + 0.5)

    t = [-xmin, -ymin]
    Ht = np.array([[1, 0, t[0]], [0, 1, t[1]], [0, 0, 1]])

    warped_img2 = cv2.warpPerspective(img2, Ht @ H, (xmax - xmin, ymax - ymin))
    warped_img2[t[1]:h1 + t[1], t[0]:w1 + t[0]] = img1

    return warped_img2

def blend_images(img1, img2):
    mask = np.where(img1 != 0, 1, 0).astype(np.float32)
    blended_img = img1 * mask + img2 * (1 - mask)
    return blended_img.astype(np.uint8)
    
def stitcher(image1 , image2):
    # Convert images to grayscale
    gray1 = cv2.cvtColor(image1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)
    
    # Initialize the feature detector and extractor (e.g., SIFT)
    sift = cv2.SIFT_create()
    
    # Detect keypoints and compute descriptors for both images
    keypoints1, descriptors1 = sift.detectAndCompute(gray1, None)
    keypoints2, descriptors2 = sift.detectAndCompute(gray2, None)
    
    # Initialize the feature matcher using brute-force matching
    bf = cv2.BFMatcher()
    
    # Match the descriptors using brute-force matching
    matches = bf.match(descriptors1, descriptors2)
    
    # Select the top N matches
    num_matches = 50
    matches = sorted(matches, key=lambda x: x.distance)[:num_matches]
    
    # Extract matching keypoints
    src_points = np.float32([keypoints1[match.queryIdx].pt for match in matches]).reshape(-1, 1, 2)
    dst_points = np.float32([keypoints2[match.trainIdx].pt for match in matches]).reshape(-1, 1, 2)
    
    # Estimate the homography matrix
    homography, _ = cv2.findHomography(src_points, dst_points, cv2.RANSAC, 5.0)
    
    # Warp the first image using the homography

    result = cv2.warpPerspective(image1, homography, ( image2.shape[1] + image1.shape[1] ,image2.shape[0]+image1.shape[0]))
    
    #result = blend_images(warped_img, image1)
    # Blending the warped image with the second image using alpha blending
    alpha = 0.5  # blending factor
    #blended_image = cv2.addWeighted(result, alpha, image2, 1 - alpha, 0)
    result[0:image2.shape[0], 0:image2.shape[1]] = image2
    result = result[0:image2.shape[0] + image1.shape[0] , 0 :image2.shape[1] + image1.shape[1] ]
    
    return result




bs = Images[0]
#fig, (ax1,ax2) = plt.subplots(nrows=1, ncols=2)
for i in range(1,len(Images)-1):
    bs = stitcher(bs,Images[i])
    #cv2.imshow("Image"+str(i), bs)
    
out = bs.copy()
# Display the blended image
cv2.imshow('Image', out)
cv2.waitKey(0)
cv2.destroyAllWindows()

[array([[[ 65,  56,  42],
        [ 61,  52,  38],
        [ 63,  54,  40],
        ...,
        [ 25,  23,  13],
        [ 25,  23,  13],
        [ 25,  23,  13]],

       [[ 70,  61,  47],
        [ 67,  58,  44],
        [ 67,  58,  44],
        ...,
        [ 25,  23,  13],
        [ 25,  23,  13],
        [ 25,  23,  13]],

       [[ 66,  57,  44],
        [ 71,  62,  49],
        [ 74,  62,  50],
        ...,
        [ 25,  22,  14],
        [ 25,  22,  14],
        [ 25,  22,  14]],

       ...,

       [[102,  78,  42],
        [ 99,  77,  41],
        [111,  89,  54],
        ...,
        [141,  98,  47],
        [131,  88,  37],
        [118,  77,  28]],

       [[ 71,  49,  14],
        [104,  82,  47],
        [103,  82,  50],
        ...,
        [136,  93,  44],
        [133,  90,  41],
        [139,  95,  48]],

       [[113,  91,  56],
        [ 97,  77,  42],
        [112,  91,  60],
        ...,
        [122,  78,  31],
        [111,  67,  20],
        [123,  79,  32]